# Tutorial 9: Neural Band (sEMG) Data

## Introduction

An Aria Gen 2 recording made while a **Meta Neural Band** was paired carries an extra `emg` stream.
The band is a wristband carrying surface electromyography (**sEMG**) electrodes plus its own
accelerometer and gyroscope. sEMG measures the tiny electrical potentials that skeletal muscle
generates when it contracts, picked up at the skin — it responds to the *intent* to move, which is
why it is used for gesture and micro-gesture work.

**This stream is optional.** Two things have to be true for it to exist: a Neural Band was paired to
the glasses, and the session used a profile that enables the band's raw EMG batch —
`profile8_emg` for recording, `profile9_emg` for streaming. Gen 2 only; there is no Gen 1
equivalent.

**The two things to understand before anything else**

1. **One VRS record is not one sample.** It is a **batch** holding many sub-samples of EMG,
   accelerometer and gyroscope data. Code written on the assumption that one record equals one
   reading will silently keep a single point per batch and throw away almost all of the signal.
2. **The band and the glasses run on two different clocks.** Every sub-sample carries a timestamp on
   each of them, and only one of the two lines up with the rest of the recording. Picking the wrong
   one puts your EMG trace seconds away from where it was recorded.

Both are covered before anything else below.

**What you'll learn:**

- How to find the `emg` stream, and why the obvious presence check is not sufficient
- The `NeuralBandBatch` structure, the two clocks its timestamps live on, and which one to use
- **Part A** — the EMG signal: reading it, converting raw ADC counts to volts, assembling a
  continuous array, and plotting it
- **Part B** — the band's IMU: accelerometer and gyroscope, and their separate calibration path
- Streaming the stream through the queued API, and lining EMG up against the glasses' own sensors

**Prerequisites**
- Complete Tutorial 1 (VrsDataProvider Basics) to understand basic data provider concepts
- Complete Tutorial 3 (Queued Sensor Data) for the sequential-access API used at the end

### ⚠️ Important Notes
- **Google Colab Users:**  
  If you encounter a `ModuleNotFoundError: No module named 'rerun'` error after installing `rerun-sdk`, Colab may not recognize the new package until the runtime is restarted.  
  **Fix:** Go to **Runtime → Restart session and run all**.

- **Visualization Issue :**  
  If a Rerun visualization window does not appear, this may be due to a known caching issue. Simply re-run the visualization cell to resolve it.

## Setup Environment (Google Colab)

If running on Google Colab, install `projectaria-tools`. This tutorial needs one input: a VRS
recorded with a Neural Band paired, so that it carries an `emg` stream.

In [ ]:
import os
import sys

google_colab_env = 'google.colab' in str(get_ipython())

if google_colab_env:
    print("Running from Google Colab, installing projectaria_tools")

    # Install projectaria-tools
    !pip install projectaria-tools==2.2.0

    # Running this command to trigger early failure of importing ReRun.
    # Should be resolved by restarting the Colab session.
    import rerun as rr

# A VRS recorded with a Meta Neural Band paired, so that it carries an `emg` stream.
emg_vrs_file_path = "path/to/your/neural_band_recording.vrs"
print(f"Using {emg_vrs_file_path}")

## Finding the `emg` stream

The stream label is `emg` (stream id `241-1`). Resolving that label is **not** on its own a check
that the recording contains Neural Band data:

- On Gen 2, `get_stream_id_from_label` answers from a mostly static device-model table. It returns
  an id for every label the glasses *could* record, so it hands back `241-1` even for a recording
  made with no band paired at all.
- A stream can also be declared in the file while carrying zero data records.

Both cases sail straight past an `is None` check and then fail on the first indexed read. Check the
record count as well.

In [ ]:
from projectaria_tools.core import data_provider

EMG_LABEL = "emg"


def find_emg_stream(provider, vrs_path):
    """
    Return (stream_id, num_batches) for the Neural Band stream, or raise.

    Checks both that the label resolves and that the stream actually holds
    records - on Gen 2 the first does not imply the second.
    """
    stream_id = provider.get_stream_id_from_label(EMG_LABEL)
    if stream_id is None:
        raise RuntimeError(f"'{EMG_LABEL}' is not a known stream label for {vrs_path}.")

    num_batches = provider.get_num_data(stream_id)
    if num_batches == 0:
        raise RuntimeError(
            f"'{EMG_LABEL}' stream is declared in {vrs_path} but carries no records. "
            "This recording was most likely made without a Neural Band paired, or with a "
            "profile that does not enable the band's raw EMG batch (profile8_emg for "
            "recording, profile9_emg for streaming)."
        )
    return stream_id, num_batches


emg_provider = data_provider.create_vrs_data_provider(emg_vrs_file_path)
emg_stream_id, num_batches = find_emg_stream(emg_provider, emg_vrs_file_path)

print(f"Opened {emg_vrs_file_path}")
print(f"  emg stream id: {emg_stream_id}")
print(f"  emg records:   {num_batches}  (records, not samples - see below)")

## Stream configuration

```
vrs_data_provider.get_neural_band_batch_configuration(stream_id)
```

**`NeuralBandBatchConfiguration` fields**

| Field Name | Description |
| :-- | :-- |
| `stream_id` | Numeric id of the `emg` stream |
| `sensor_model` | Wristband hardware model string |
| `device_id` | Band device identifier |
| `emg_calibration_params_json` | The verbatim CONFIG-record JSON blob. Carries both the EMG and the IMU calibration fields. Empty when absent |
| `emg_calibration` | Parsed EMG calibration, or `None` when absent or malformed |
| `imu_calibration` | Parsed IMU calibration, or `None` when absent or malformed |

Both calibration objects are **optional**. Part A and Part B each show what to do when theirs is
missing, so check before use rather than assuming.

In [ ]:
emg_config = emg_provider.get_neural_band_batch_configuration(emg_stream_id)

print("=== Neural Band configuration ===")
print(f"  stream_id:                    {emg_config.stream_id}")
print(f"  sensor_model:                 '{emg_config.sensor_model}'")
print(f"  device_id:                    {emg_config.device_id}")
print(f"  emg_calibration_params_json:  {len(emg_config.emg_calibration_params_json)} chars")
print(f"  emg_calibration present:      {emg_config.emg_calibration is not None}")
print(f"  imu_calibration present:      {emg_config.imu_calibration is not None}")

## The batch model

This is the part that trips people up. One VRS record in the `emg` stream is a **`NeuralBandBatch`**,
and each batch holds three independent arrays of sub-samples:

```
NeuralBandBatch
├── arrival_timestamp_ns       device clock: when the glasses took this batch off the band link
├── batch_sequence_number      monotonic counter, useful for spotting dropped batches
├── emg_channel_count          how many electrode channels each EMG sub-sample carries
├── emg_bits_per_adc_reading   ADC width of the values in channel_values
│
├── emg   [ NeuralBandEmgSample,   ... ]   channel_values (raw ADC counts)
├── accel [ NeuralBandAccelSample, ... ]   accel_msec2  (m/s^2)
└── gyro  [ NeuralBandGyroSample,  ... ]   gyro_radsec  (rad/s)
```

Every sub-sample, whatever its type, carries the same two timestamp fields:

| Field | Clock | Notes |
| :-- | :-- | :-- |
| `wristband_timestamp_ns` | Band | Straight off the wire, 1 µs resolution. Not comparable to anything else in the recording |
| `device_timestamp_ns` | Glasses | `Optional[int]` — `None` when the two clocks could not be related. **This is the one to use** |

The three arrays are independent: they have their own lengths, their own rates, and their own
timestamps. `len(batch.emg)` is typically much larger than `len(batch.accel)`.

To put numbers on it, a `profile8_emg` recording typically produces:

| | Rate | Per batch |
| :-- | :-- | :-- |
| Batches | ~128 Hz | — |
| EMG | ~2048 Hz | 16 sub-samples |
| Accelerometer | ~128 Hz | 1 sub-sample |
| Gyroscope | ~128 Hz | 1 sub-sample |

So `get_num_data(emg_stream_id)` returns a count of **batches**, not of EMG samples — and at these
rates it undercounts the EMG signal by a factor of about 16. The real sample count is the sum of
`len(batch.emg)` across every batch. Treat the table as indicative rather than guaranteed; the cell
below reads the actual numbers out of your recording.

In [ ]:
sample_batch = emg_provider.get_neural_band_batch_by_index(emg_stream_id, 0)


def describe_span(sub_samples):
    """Report the span of a sub-sample array on each of the two clocks."""
    first, last = sub_samples[0], sub_samples[-1]
    wristband_ms = (last.wristband_timestamp_ns - first.wristband_timestamp_ns) / 1e6
    print(f"    wristband_timestamp_ns: {first.wristband_timestamp_ns} .. "
          f"{last.wristband_timestamp_ns}  (span {wristband_ms:.3f} ms)")
    if first.device_timestamp_ns is None or last.device_timestamp_ns is None:
        print("    device_timestamp_ns:    None - the clocks could not be related")
        return
    device_ms = (last.device_timestamp_ns - first.device_timestamp_ns) / 1e6
    print(f"    device_timestamp_ns:    {first.device_timestamp_ns} .. "
          f"{last.device_timestamp_ns}  (span {device_ms:.3f} ms)")


print("=== One NeuralBandBatch, dissected ===")
print(f"  arrival_timestamp_ns:     {sample_batch.arrival_timestamp_ns}")
print(f"  batch_sequence_number:    {sample_batch.batch_sequence_number}")
print(f"  emg_channel_count:        {sample_batch.emg_channel_count}")
print(f"  emg_bits_per_adc_reading: {sample_batch.emg_bits_per_adc_reading}")

for name, sub_samples in (
    ("emg", sample_batch.emg),
    ("accel", sample_batch.accel),
    ("gyro", sample_batch.gyro),
):
    print(f"\n  {name}: {len(sub_samples)} sub-samples in this one batch")
    if sub_samples:
        describe_span(sub_samples)

print(f"\n  first EMG sub-sample channel_values: {sample_batch.emg[0].channel_values}")
print(f"    ^ {len(sample_batch.emg[0].channel_values)} raw ADC counts, one per electrode channel")

# How much signal is actually in this stream, as opposed to how many records.
total_emg_samples = sum(
    len(emg_provider.get_neural_band_batch_by_index(emg_stream_id, i).emg)
    for i in range(min(num_batches, 50))
)
print(f"\n  {total_emg_samples} EMG samples in the first {min(num_batches, 50)} batches alone")

### The two clocks

Read this before using any Neural Band timestamp. Nothing else in an Aria recording works this way.

The band samples against **its own crystal** and the glasses stamp against **theirs**. The two
free-run: there is no hardware sync between them, and their offset drifts over a session. Three
distinct times are therefore in play.

**`batch.arrival_timestamp_ns` — device clock, but not a sampling instant.** The moment the *glasses*
took this batch off the band link. It is a receive time, and it inherits the link's burst arrival
pattern, so consecutive batches are spaced far less regularly than the samples inside them. The VRS
index is ordered by it, so it is what a time query resolves against — but do not plot signal on it.
`batch_sequence_number` is the regular quantity.

**`sample.wristband_timestamp_ns` — band clock, as it came off the wire.** Even and trustworthy to a
microsecond *relative to other band samples*, and meaningless against anything else in the recording:
it is a different timebase, typically seconds away from device time. One more wrinkle for EMG: the
wire carries **one timestamp per packet**, not per sub-sample, so every sub-sample of a packet
repeats it. On shipping firmware a batch is a single packet, which is why the EMG wristband span
printed above is `0.000 ms` across all 16 sub-samples. Treat it as a packet label, not a sample time.

**`sample.device_timestamp_ns` — the one you want.** The same instant expressed on the device clock,
so it is directly comparable to RGB frames, glasses IMU and everything else in the file. Producing it
takes two steps, both done for you, with no flag to set and no time-sync stream required:

- *Relating the clocks.* Every record states a (wristband, arrival) pair. A single pair is a poor
  estimate — reception is bursty — so the offset is fitted as a median over a window of records at
  fixed positions in the file, and a conversion interpolates between the two anchors bracketing it.
  Anchor positions depend only on the file, so a given timestamp converts identically however you
  reached it.
- *Spreading the EMG sub-samples.* The packet's wire timestamp is the instant of its **first**
  sub-sample, so the rest step forward from it at the measured sample period. This is where EMG stops
  being a step function.

**`device_timestamp_ns` is `Optional`, and `None` is not a failure to route around.** It means no
mapping could be produced, and the wristband clock is *not* a fallback — plotting on it would put the
trace somewhere it was not recorded. Drop the samples instead. It comes back `None` when:

- The recording is too short, or too many of its records are unreadable, to fit an anchor. A fit
  needs at least 8 readable records inside its window, so an `emg` stream of roughly a second or
  less produces no device timestamps at all.
- The EMG cadence probe could not run, in which case accel and gyro can still be mapped while EMG is
  not — so check per sub-stream rather than once.

**What this buys you, and what it does not.** Relative timing within the band's own data is good, and
now so is absolute placement against the glasses, to the accuracy of the clock fit. What no timestamp
in this stream can tell you is transport latency: the muscle activity happened before the glasses
received it, and nothing in the recording measures by how much. Do not read a Neural Band timestamp
as the instant the muscle fired.

In [ ]:
# The sample period is measured from the recording - the band's own crystal, not a
# hardware constant - so it is read back rather than assumed.
emg_period_ns = emg_provider.get_neural_band_emg_sample_period_ns(emg_stream_id)
if emg_period_ns is None:
    print("EMG sample period unavailable: too few readable records to measure the cadence.")
else:
    print(f"Measured EMG sample period: {emg_period_ns} ns ({1e9 / emg_period_ns:.1f} Hz)")

# The two clocks are also exposed directly, for when you have a bare timestamp rather
# than a sample. Both return None under the same conditions as device_timestamp_ns.
# Note there is no TimeDomain for the wristband clock - it is not a sync domain, and
# these two calls are the only way across.
probe_wristband_ns = sample_batch.emg[0].wristband_timestamp_ns
mapped_ns = emg_provider.convert_from_wristband_time_to_device_time_ns(
    probe_wristband_ns, emg_stream_id
)
print(f"\nwristband {probe_wristband_ns} ns -> device {mapped_ns} ns")

if mapped_ns is not None:
    round_trip_ns = emg_provider.convert_from_device_time_to_wristband_time_ns(
        mapped_ns, emg_stream_id
    )
    print(f"  and back: {round_trip_ns} ns "
          f"(off by {abs(round_trip_ns - probe_wristband_ns)} ns)")
    print(f"  clock offset here: {(mapped_ns - probe_wristband_ns) / 1e9:.3f} s")
    print("  ^ the size of the mistake plotting on the wristband clock would make")

---

# Part A — The EMG signal

## A.1 Reading batches

The `emg` stream is queried like any other sensor stream:

- `get_neural_band_batch_by_index(stream_id, index)`
- `get_neural_band_batch_by_time_ns(stream_id, time_ns, time_domain, query_options)`

Two things to keep in mind for the time-based query. It returns **the whole batch**, not the single
sub-sample nearest your query time — once you have the batch, find the sub-sample you want inside it.
And the batch it picks is selected by `arrival_timestamp_ns`, the only timestamp the VRS index is
ordered by, so a query time lands *near* the samples it returns rather than exactly among them.

In [ ]:
from projectaria_tools.core.sensor_data import TimeDomain, TimeQueryOptions

# By index.
batch_by_index = emg_provider.get_neural_band_batch_by_index(emg_stream_id, min(5, num_batches - 1))
print(f"By index: batch #{batch_by_index.batch_sequence_number} "
      f"arrived at {batch_by_index.arrival_timestamp_ns} ns, "
      f"{len(batch_by_index.emg)} EMG sub-samples")

# By timestamp. BEFORE gives the most recent batch at or before the query time.
query_time_ns = batch_by_index.arrival_timestamp_ns
batch_by_time = emg_provider.get_neural_band_batch_by_time_ns(
    emg_stream_id, query_time_ns, TimeDomain.DEVICE_TIME, TimeQueryOptions.BEFORE
)
print(f"By time:  batch #{batch_by_time.batch_sequence_number} "
      f"arrived at {batch_by_time.arrival_timestamp_ns} ns")

# The batch was selected on arrival time; its samples sit before it, because the band
# sampled them before the glasses received them.
device_times = [s.device_timestamp_ns for s in batch_by_time.emg]
if device_times and all(t is not None for t in device_times):
    print(f"  its EMG sub-samples span {device_times[0]} .. {device_times[-1]} ns "
          "on the device clock")
    print(f"  arrival is {(query_time_ns - device_times[-1]) / 1e6:.3f} ms "
          "after the last of them")
else:
    print("  its EMG sub-samples carry no device timestamps in this recording")

## A.2 Raw ADC counts are not volts

`NeuralBandEmgSample.channel_values` holds **raw ADC counts**. This is the single most common source
of confusion in this stream, and it is worth contrasting with the band's IMU in Part B, whose
accelerometer and gyroscope values arrive already converted to physical units.

To get volts at the electrode input you need the EMG calibration, which is reachable two ways:

1. **From the stream configuration** — `config.emg_calibration`
2. **From the device calibration** — `provider.get_sensor_calibration(stream_id)`, check
   `sensor_calibration_type() == SensorCalibrationType.NEURAL_BAND_BATCH_CALIBRATION`, then
   `neural_band_batch_calibration()` gives you an object with `.emg_calib` and `.imu_calib`

Both routes reach the same `NeuralBandEmgCalibration`. Use whichever fits where you already are.

**`NeuralBandEmgCalibration`**

| Method | Description |
| :-- | :-- |
| `get_label()` | Calibration label |
| `get_analog_gain()` | Analog front-end gain |
| `get_adc_count_levels()` | Number of ADC quantization levels |
| `get_adc_vref_pos()`, `get_adc_vref_neg()`, `get_adc_vref_dc()` | ADC reference voltages |
| `is_truncated()` | Selects which conversion formula applies. `adc_vref_dc` is subtracted only in the non-truncated formula |
| `get_streamed_bit_width()` | Bit width of the values actually streamed |
| `get_dropped_lsb()` | Low bits dropped before streaming |
| `get_adc_chip_code()` | Opaque ADC front-end identifier. `0` means unknown; other codes are hardware-specific and intentionally not decoded |
| `adc_to_volts(adc_count)` | Convert one count |
| `adc_to_volts(adc_counts)` | Convert a whole list in one call |
| `volts_to_adc(volts)` | Round-to-nearest inverse |
| `from_params_json(json, label)` | Parse a calibration out of the raw JSON blob yourself |

:::caution `adc_to_volts` range-checks on one branch only
Which branch you are on is `is_truncated()`. When it is `True`, a count at or above
`2 ** get_streamed_bit_width()` raises `ValueError`. When it is `False` there is no range check at
all: the formula extrapolates past the ADC's own full scale without complaint and hands back a
plausible-looking voltage. So the same out-of-range count is a loud failure on one calibration and a
silent wrong answer on another — if you are converting values you did not read straight out of a
batch, range-check them yourself rather than relying on either behaviour.
:::

In [ ]:
from projectaria_tools.core.calibration import SensorCalibrationType

# Route 1: straight off the stream configuration.
emg_calib = emg_config.emg_calibration

# Route 2: via the device calibration. Shown because it is the route you already
# have in hand when you are working with the rest of the device calibration.
sensor_calib = emg_provider.get_sensor_calibration(emg_stream_id)
if (
    sensor_calib is not None
    and sensor_calib.sensor_calibration_type() == SensorCalibrationType.NEURAL_BAND_BATCH_CALIBRATION
):
    batch_calib = sensor_calib.neural_band_batch_calibration()
    print("Reached NeuralBandBatchCalibration via get_sensor_calibration()")
    print(f"  emg_calib present: {batch_calib.emg_calib is not None}")
    print(f"  imu_calib present: {batch_calib.imu_calib is not None}")
    if emg_calib is None:
        emg_calib = batch_calib.emg_calib

if emg_calib is None:
    print("\nNo EMG calibration in this recording - channel_values stay as raw ADC counts.")
else:
    print("\n=== NeuralBandEmgCalibration ===")
    print(f"  label:              '{emg_calib.get_label()}'")
    print(f"  analog_gain:        {emg_calib.get_analog_gain()}")
    print(f"  adc_count_levels:   {emg_calib.get_adc_count_levels()}")
    print(f"  adc_vref_pos:       {emg_calib.get_adc_vref_pos()}")
    print(f"  adc_vref_neg:       {emg_calib.get_adc_vref_neg()}")
    print(f"  adc_vref_dc:        {emg_calib.get_adc_vref_dc()}")
    print(f"  is_truncated:       {emg_calib.is_truncated()}")
    print(f"  streamed_bit_width: {emg_calib.get_streamed_bit_width()}")
    print(f"  dropped_lsb:        {emg_calib.get_dropped_lsb()}")
    print(f"  adc_chip_code:      {emg_calib.get_adc_chip_code()}")

In [ ]:
if emg_calib is None:
    print("No EMG calibration - skipping the volts conversion.")
else:
    first_sample = sample_batch.emg[0]

    # Scalar form: one count at a time.
    single_adc = first_sample.channel_values[0]
    print(f"adc_to_volts({single_adc}) = {emg_calib.adc_to_volts(single_adc):.9f} V")

    # List form: hand it the whole channel vector at once. This is the form to use
    # in a loop - one call across a flat list, then reshape, rather than a Python
    # call per value.
    volts = emg_calib.adc_to_volts(first_sample.channel_values)
    print(f"adc_to_volts(channel_values) = {[f'{v:.6f}' for v in volts]}")

    # Round-trip. Exact only for voltages that land on the ADC grid.
    back = emg_calib.volts_to_adc(emg_calib.adc_to_volts(single_adc))
    print(f"volts_to_adc round-trip: {single_adc} -> {back}  (exact: {back == single_adc})")

    # What an out-of-range count does depends on which formula this calibration selects.
    if emg_calib.is_truncated():
        beyond_full_scale = 1 << emg_calib.get_streamed_bit_width()
        try:
            emg_calib.adc_to_volts(beyond_full_scale)
        except ValueError as error:
            print(f"\nadc_to_volts({beyond_full_scale}) raised: {error}")
            print("  ^ truncated calibration: counts past the streamed bit width are rejected")
    else:
        beyond_full_scale = emg_calib.get_adc_count_levels() + 1
        print(f"\nadc_to_volts({beyond_full_scale}) = "
              f"{emg_calib.adc_to_volts(beyond_full_scale):.9f} V")
        print("  ^ non-truncated calibration: past full scale, extrapolated without complaint")

## A.3 Assembling a continuous array

Most analysis wants one contiguous time series, not a list of batches. Flattening the sub-samples
out of every batch is the step you will write in nearly every script that touches this stream, so it
is worth having in one place.

The result is:

- `emg_timestamps_ns` — shape `(num_samples,)`, on the **device** clock
- `emg_adc` — shape `(num_samples, num_channels)`, raw counts
- `emg_volts` — same shape, in volts

Two things the loader does deliberately. It reads `device_timestamp_ns` and **drops** any sub-sample
where it is `None`, rather than substituting the wristband timestamp — the two are different
timebases, and mixing them would corrupt the series silently. And the volts conversion is one
`adc_to_volts` call over the whole flattened list, reshaped afterwards; calling it per sample would
cross the Python/C++ boundary once per reading, which at EMG rates is the difference between a fast
cell and an unusably slow one.

In [ ]:
import numpy as np

# Everything below works on the first VIZ_DURATION_SEC of the recording. Raise it
# for real analysis; it is bounded here only to keep the cells responsive.
VIZ_DURATION_SEC = 10


def load_emg_array(provider, stream_id, num_batches, calibration=None, duration_sec=None):
    """
    Flatten the EMG sub-samples of consecutive batches into contiguous arrays.

    Starts at the first batch. When `duration_sec` is set, stops once the loaded
    sub-samples span that much time, so how much is read does not depend on the
    band's batch rate.

    Timestamps come out on the device clock. Sub-samples without one are dropped:
    the wristband clock they also carry is a different timebase, so falling back
    to it would place them seconds away from the rest of the recording.

    Returns (timestamps_ns, adc, volts). `volts` is None when no calibration was
    supplied. Batches whose channel vectors do not match the batch's declared
    channel count are skipped rather than ragged-ing the array.
    """
    limit_ns = None if duration_sec is None else int(duration_sec * 1e9)

    timestamps, rows = [], []
    channel_count = None
    skipped_shape = 0
    skipped_unmapped = 0
    start_ns = None

    for index in range(num_batches):
        batch = provider.get_neural_band_batch_by_index(stream_id, index)
        if not batch.emg or batch.emg_channel_count == 0:
            continue
        if channel_count is None:
            channel_count = batch.emg_channel_count

        for sub_sample in batch.emg:
            if len(sub_sample.channel_values) != channel_count:
                skipped_shape += 1
                continue
            device_ns = sub_sample.device_timestamp_ns
            if device_ns is None:
                skipped_unmapped += 1
                continue
            if start_ns is None:
                start_ns = device_ns
            timestamps.append(device_ns)
            rows.append(sub_sample.channel_values)

        if limit_ns is not None and timestamps and timestamps[-1] - start_ns >= limit_ns:
            break

    if skipped_shape:
        print(f"  skipped {skipped_shape} sub-samples with an unexpected channel count")
    if skipped_unmapped:
        print(f"  skipped {skipped_unmapped} sub-samples with no device timestamp")
    if not rows:
        raise RuntimeError(
            "No usable EMG sub-samples found. If every sub-sample was dropped for "
            "having no device timestamp, the recording is too short or too damaged "
            "to relate the band and glasses clocks."
        )

    timestamps_ns = np.asarray(timestamps, dtype=np.int64)
    adc = np.asarray(rows, dtype=np.int64)

    volts = None
    if calibration is not None:
        # One vectorized call over the flattened counts, then reshape back.
        flat = [value for row in rows for value in row]
        volts = np.asarray(calibration.adc_to_volts(flat), dtype=np.float64).reshape(adc.shape)

    return timestamps_ns, adc, volts


emg_timestamps_ns, emg_adc, emg_volts = load_emg_array(
    emg_provider, emg_stream_id, num_batches, calibration=emg_calib,
    duration_sec=VIZ_DURATION_SEC,
)

print(f"Loaded {len(emg_timestamps_ns)} EMG samples from the start of the recording")
print(f"  emg_timestamps_ns: {emg_timestamps_ns.shape}  device clock")
print(f"  emg_adc:           {emg_adc.shape}  raw counts, "
      f"range [{emg_adc.min()}, {emg_adc.max()}]")
if emg_volts is not None:
    print(f"  emg_volts:         {emg_volts.shape}  V, "
          f"range [{emg_volts.min():.6f}, {emg_volts.max():.6f}]")

covered_sec = (emg_timestamps_ns[-1] - emg_timestamps_ns[0]) / 1e9
print(f"  covering {covered_sec:.3f} s from the start of the stream")

# Sanity check on the spread: consecutive sub-samples should step by the measured
# period rather than repeating one packet timestamp.
median_step_ns = float(np.median(np.diff(emg_timestamps_ns)))
if median_step_ns > 0:
    print(f"  median sub-sample step: {median_step_ns:.0f} ns "
          f"({1e9 / median_step_ns:.1f} Hz)")
else:
    print("  median sub-sample step: 0 ns - the sub-samples were not spread")

## A.4 Plotting EMG

A note on how, because the naive version does not scale. EMG carries thousands of samples per
second per channel, and logging them to Rerun one at a time — an `rr.set_time` plus an `rr.log` per
sample — makes even a few seconds of signal take minutes to draw.

Use **`rr.send_columns`** instead: it hands Rerun an entire column of timestamps and an entire
column of values in a single call, per channel. Per-channel color and legend name are set once with
a `static=True` `rr.SeriesLines` log.

In [ ]:
import colorsys

import rerun as rr

EMG_ADC_PATH = "neural-band-emg"
EMG_VOLTS_PATH = "neural-band-emg-volts"


def log_emg_series(entity_root, timestamps_ns, values, label_suffix=""):
    """Log one Rerun scalar series per EMG channel, in a single call per channel."""
    # `device_time` has to stay a duration timeline. The RGB cells below set it with
    # `duration=`, and Rerun rejects a chunk whose timeline changes type, so mixing in
    # a `timestamp=` column makes the combined view drop its data. `timestamp=` would
    # also read these as seconds since the Unix epoch, which device time is not.
    timestamps = timestamps_ns.astype("timedelta64[ns]")
    channel_count = values.shape[1]

    for channel in range(channel_count):
        hue = channel / max(channel_count, 1)
        r, g, b = colorsys.hsv_to_rgb(hue, 0.85, 0.9)
        rr.log(
            f"{entity_root}/channels/channel_{channel}",
            rr.SeriesLines(
                colors=[[int(r * 255), int(g * 255), int(b * 255)]],
                names=[f"channel_{channel}{label_suffix}"],
            ),
            static=True,
        )
        rr.send_columns(
            f"{entity_root}/channels/channel_{channel}",
            indexes=[rr.TimeColumn("device_time", duration=timestamps)],
            columns=rr.Scalars.columns(scalars=values[:, channel].astype(np.float64)),
        )


rr.init("rerun_viz_neural_band_emg")

log_emg_series(EMG_ADC_PATH, emg_timestamps_ns, emg_adc, " (ADC)")
if emg_volts is not None:
    log_emg_series(EMG_VOLTS_PATH, emg_timestamps_ns, emg_volts, " (V)")

print(f"Logged {emg_adc.shape[1]} channels x {emg_adc.shape[0]} samples")
rr.notebook_show()

---

# Part B — The band's IMU

The same batches also carry the wristband's own accelerometer and gyroscope. These are useful on
their own for wrist orientation and gross arm motion, and they come from the band, so they are
distinct from the two IMUs in the glasses.

**Units work the opposite way from EMG.** Where `channel_values` needed converting, these arrive
already in physical units:

- `NeuralBandAccelSample.accel_msec2` — xyz in m/s²
- `NeuralBandGyroSample.gyro_radsec` — xyz in rad/s

There is a subtlety behind that convenience. The scale factor used to get there depends on whether
the recording carried an IMU calibration: when `imu_calibration` is present its accelerometer and
gyroscope scaling factors are used, and when it is absent a default constant is used instead. The
values are in physical units either way, but they are better trusted when the calibration was
present — which is worth checking rather than assuming.

**Timestamps work the same way as EMG, with one simplification.** Accel and gyro get their
`device_timestamp_ns` from the same clock fit, but they need no sub-sample spreading — the wire
carries a real timestamp per IMU sample, not one per packet. That also means they can be mapped in a
recording where EMG is not, so check each sub-stream on its own. Note too that accel and gyro run on
their own trigger, a few hundred microseconds off the EMG in the same batch; they share a batch, not
an instant.

In [ ]:
imu_calib = emg_config.imu_calibration

print(f"IMU calibration present: {imu_calib is not None}")
if imu_calib is None:
    print("  Falling back to default scale factors for accel/gyro.")
else:
    print("=== NeuralBandImuCalibration ===")
    print(f"  label:                 '{imu_calib.get_label()}'")
    print(f"  accel_scaling_factor:  {imu_calib.get_accel_scaling_factor()}")
    print(f"  gyro_scaling_factor:   {imu_calib.get_gyro_scaling_factor()}")
    print(f"  calibration_applied:   {imu_calib.get_calibration_applied().name}")
    print(f"  offline accel offset (g):   {imu_calib.get_offline_accel_offset_g()}")
    print(f"  offline gyro offset (dps):  {imu_calib.get_offline_gyro_offset_dps()}")
    print(f"  online  accel offset (g):   {imu_calib.get_online_accel_offset_g()}")
    print(f"  online  gyro offset (dps):  {imu_calib.get_online_gyro_offset_dps()}")

    # Cross-axis rectification, for going from raw axes to a rectified frame.
    print(f"  accel cross-axis matrix:\n{imu_calib.get_accel_cross_axis_rect_matrix()}")

    raw_accel = np.asarray(sample_batch.accel[0].accel_msec2, dtype=np.float64)
    rectified = imu_calib.raw_to_rectified_accel(raw_accel)
    print(f"\n  raw_to_rectified_accel({raw_accel}) = {rectified}")

In [ ]:
def load_band_imu_arrays(provider, stream_id, num_batches, duration_sec=None):
    """
    Flatten the accel and gyro sub-samples out of consecutive batches.

    Starts at the first batch and, like load_emg_array, stops once the loaded
    sub-samples span `duration_sec`, and drops sub-samples with no device
    timestamp.
    """
    limit_ns = None if duration_sec is None else int(duration_sec * 1e9)

    accel_ts, accel_xyz, gyro_ts, gyro_xyz = [], [], [], []
    unmapped = 0
    start_ns = None
    for index in range(num_batches):
        batch = provider.get_neural_band_batch_by_index(stream_id, index)
        for sub_sample in batch.accel:
            if sub_sample.device_timestamp_ns is None:
                unmapped += 1
                continue
            if start_ns is None:
                start_ns = sub_sample.device_timestamp_ns
            accel_ts.append(sub_sample.device_timestamp_ns)
            accel_xyz.append(sub_sample.accel_msec2)
        for sub_sample in batch.gyro:
            if sub_sample.device_timestamp_ns is None:
                unmapped += 1
                continue
            gyro_ts.append(sub_sample.device_timestamp_ns)
            gyro_xyz.append(sub_sample.gyro_radsec)

        if limit_ns is not None and accel_ts and accel_ts[-1] - start_ns >= limit_ns:
            break

    if unmapped:
        print(f"  skipped {unmapped} IMU sub-samples with no device timestamp")
    return (
        np.asarray(accel_ts, dtype=np.int64),
        np.asarray(accel_xyz, dtype=np.float64),
        np.asarray(gyro_ts, dtype=np.int64),
        np.asarray(gyro_xyz, dtype=np.float64),
    )


accel_ts_ns, accel_xyz, gyro_ts_ns, gyro_xyz = load_band_imu_arrays(
    emg_provider, emg_stream_id, num_batches, duration_sec=VIZ_DURATION_SEC
)
print(f"accel: {accel_xyz.shape}  m/s^2")
print(f"gyro:  {gyro_xyz.shape}  rad/s")

AXIS_COLORS = [[255, 80, 80], [80, 255, 80], [80, 120, 255]]

rr.init("rerun_viz_neural_band_imu")
for root, timestamps, values in (
    ("neural-band-accel", accel_ts_ns, accel_xyz),
    ("neural-band-gyro", gyro_ts_ns, gyro_xyz),
):
    if len(timestamps) == 0:
        print(f"{root}: no sub-samples in this recording")
        continue
    # Duration, not timestamp - see the note in the EMG series helper above.
    timestamps_dt = timestamps.astype("timedelta64[ns]")
    for axis, axis_name in enumerate("xyz"):
        rr.log(
            f"{root}/{axis_name}",
            rr.SeriesLines(colors=[AXIS_COLORS[axis]], names=[f"{root}_{axis_name}"]),
            static=True,
        )
        rr.send_columns(
            f"{root}/{axis_name}",
            indexes=[rr.TimeColumn("device_time", duration=timestamps_dt)],
            columns=rr.Scalars.columns(scalars=values[:, axis]),
        )

rr.notebook_show()

---

# Sequential access and alignment with the glasses

## Streaming the stream

Everything above pulled batches by index. For a real pass over a recording, use the queued API from
`Tutorial_3_sequential_access_multi_sensor_data`, which delivers records from several streams in
timestamp order. Neural Band records arrive as `SensorDataType.NEURAL_BAND_BATCH` and the batch
comes out of `sensor_data.neural_band_batch_data()`.

In [ ]:
from projectaria_tools.core.sensor_data import SensorDataType

deliver_options = emg_provider.get_default_deliver_queued_options()
deliver_options.deactivate_stream_all()
deliver_options.activate_stream(emg_stream_id)

# Start at the beginning of the recording and cap the window, so no first-side
# truncation is needed.
total_length_ns = emg_provider.get_last_time_ns_all_streams(
    TimeDomain.DEVICE_TIME
) - emg_provider.get_first_time_ns_all_streams(TimeDomain.DEVICE_TIME)
duration_ns = min(int(VIZ_DURATION_SEC * 1e9), total_length_ns)
deliver_options.set_truncate_last_device_time_ns(max(total_length_ns - duration_ns, 0))

batches_seen = 0
emg_samples_seen = 0
for sensor_data in emg_provider.deliver_queued_sensor_data(deliver_options):
    if sensor_data.sensor_data_type() != SensorDataType.NEURAL_BAND_BATCH:
        continue
    batch = sensor_data.neural_band_batch_data()
    batches_seen += 1
    emg_samples_seen += len(batch.emg)

print(f"Over the first {duration_ns / 1e9:.1f} s of the recording:")
print(f"  {batches_seen} batches delivered")
print(f"  {emg_samples_seen} EMG samples inside them")

## Lining EMG up against the glasses

Once you are working in `device_timestamp_ns`, a Neural Band sample sits on the same clock as every
other stream in the recording, so it can be used to query them directly — no conversion step.

Two caveats survive that.

**Do not query with `arrival_timestamp_ns`.** It is on the device clock, so nothing will complain,
but it is when the *glasses received the batch*, not when the band sampled it. Use the
`device_timestamp_ns` of the sub-sample you actually care about; the cell below shows the gap between
the two.

**Transport latency is still unmeasured.** The clock fit places the band's samples correctly on the
device timeline as the band stamped them, but the band stamps at its own end of the link and nothing
in the recording records how long the muscle activity took to reach the electrodes and the ADC. For
labelling a gesture against video or cutting clips around an event this does not matter; for claiming
millisecond-true correspondence between muscle activity and a frame, it does.

In [ ]:
RGB_CAMERA_LABEL = "camera-rgb"

rgb_stream_id = emg_provider.get_stream_id_from_label(RGB_CAMERA_LABEL)
if rgb_stream_id is None or emg_provider.get_num_data(rgb_stream_id) == 0:
    print(f"This recording has no {RGB_CAMERA_LABEL} frames - skipping the alignment example.")
else:
    probe_batch = emg_provider.get_neural_band_batch_by_index(
        emg_stream_id, min(20, num_batches - 1)
    )
    # The sub-sample's own device time, not the batch's arrival time.
    probe_time_ns = probe_batch.emg[0].device_timestamp_ns

    if probe_time_ns is None:
        print("This batch's EMG carries no device timestamp - skipping the alignment example.")
    else:
        rgb_data_and_record = emg_provider.get_image_data_by_time_ns(
            rgb_stream_id, probe_time_ns, TimeDomain.DEVICE_TIME, TimeQueryOptions.CLOSEST
        )
        rgb_time_ns = rgb_data_and_record[1].capture_timestamp_ns

        print(f"EMG sub-sample sampled at:     {probe_time_ns} ns")
        print(f"Its batch arrived at:          {probe_batch.arrival_timestamp_ns} ns")
        print(f"  link + batching delay:       "
              f"{(probe_batch.arrival_timestamp_ns - probe_time_ns) / 1e6:.2f} ms")
        print(f"Closest RGB frame captured at: {rgb_time_ns} ns")
        print(f"  difference:                  {abs(rgb_time_ns - probe_time_ns) / 1e6:.2f} ms")
        print("\nThat last difference is dominated by the RGB frame interval - the nearest")
        print("frame is at most half a frame away. It is not a clock error.")

In [ ]:
# EMG and RGB on one Rerun timeline: scrub the video and watch the signal move with it.
if rgb_stream_id is None or emg_provider.get_num_data(rgb_stream_id) == 0:
    print(f"No {RGB_CAMERA_LABEL} frames - skipping the combined visualization.")
else:
    rr.init("rerun_viz_neural_band_with_rgb")

    # EMG first, as columns.
    log_emg_series(EMG_ADC_PATH, emg_timestamps_ns, emg_adc, " (ADC)")

    # Then RGB frames over the same span the EMG arrays cover.
    rgb_options = emg_provider.get_default_deliver_queued_options()
    rgb_options.deactivate_stream_all()
    rgb_options.activate_stream(rgb_stream_id)

    # Match the RGB window to the span the EMG arrays actually cover rather than
    # to VIZ_DURATION_SEC. The band link takes a moment to ramp up, so the first
    # EMG batch can land after the first RGB frame; following the EMG span keeps
    # the two aligned instead of assuming both streams start together.
    first_all_ns = emg_provider.get_first_time_ns_all_streams(TimeDomain.DEVICE_TIME)
    last_all_ns = emg_provider.get_last_time_ns_all_streams(TimeDomain.DEVICE_TIME)
    rgb_options.set_truncate_first_device_time_ns(
        max(int(emg_timestamps_ns[0]) - first_all_ns, 0)
    )
    rgb_options.set_truncate_last_device_time_ns(
        max(last_all_ns - int(emg_timestamps_ns[-1]), 0)
    )

    frames = 0
    for sensor_data in emg_provider.deliver_queued_sensor_data(rgb_options):
        device_time_ns = sensor_data.get_time_ns(TimeDomain.DEVICE_TIME)
        image_data_and_record = sensor_data.image_data_and_record()
        rr.set_time("device_time", duration=np.timedelta64(device_time_ns, "ns"))
        # A full-resolution Gen 2 RGB frame decodes to ~15 MB, so a 10 s window is
        # ~1.4 GB uncompressed. The notebook viewer runs in wasm with a 1 GiB budget
        # and silently drops the oldest data past it, so the start of the clip loses
        # its images. JPEG keeps the same window roughly 25x smaller.
        rr.log(
            RGB_CAMERA_LABEL,
            rr.Image(image_data_and_record[0].to_numpy_array()).compress(
                jpeg_quality=90
            ),
        )
        frames += 1

    print(f"Logged {frames} RGB frames alongside {emg_adc.shape[0]} EMG samples")
    rr.notebook_show()

---

# Summary

**The batch model**

| | |
| :-- | :-- |
| One VRS record | One `NeuralBandBatch` |
| One batch | Many EMG, accel and gyro sub-samples, in three independent arrays |
| `get_num_data(emg_stream_id)` | A count of batches, not of samples |
| Real EMG sample count | `sum(len(batch.emg))` over all batches |

**Units**

| Field | Arrives as | You must |
| :-- | :-- | :-- |
| `NeuralBandEmgSample.channel_values` | Raw ADC counts | Convert with `NeuralBandEmgCalibration.adc_to_volts` |
| `NeuralBandAccelSample.accel_msec2` | m/s² | Nothing, but check `imu_calibration` was present |
| `NeuralBandGyroSample.gyro_radsec` | rad/s | Nothing, but check `imu_calibration` was present |

**Timestamps** — three of them, and only one is for analysis

| Field | Clock | Use it for |
| :-- | :-- | :-- |
| `batch.arrival_timestamp_ns` | Device | Time queries, which the VRS index resolves on. Bursty; never plot on it |
| `sample.wristband_timestamp_ns` | Band | Nothing, on its own. For EMG it is a packet label — every sub-sample of a packet repeats it |
| `sample.device_timestamp_ns` | Device | **Everything else.** `Optional` — `None` means no mapping, so drop the sample |

- The band and the glasses run on independent, free-running clocks. `device_timestamp_ns` is the
  result of fitting the offset between them over a window of records, plus, for EMG, spreading each
  packet's sub-samples forward from its first at the measured sample period.
- `None` is not a prompt to fall back to `wristband_timestamp_ns`. Different timebase, seconds apart.
  It shows up on short or damaged recordings, and independently per sub-stream.
- `get_neural_band_emg_sample_period_ns(stream_id)` returns the period measured from the recording.
  `convert_from_wristband_time_to_device_time_ns` / `convert_from_device_time_to_wristband_time_ns`
  do the same conversion for a bare timestamp. All three return `None` under the same conditions.
- Transport latency between muscle and glasses is not measured anywhere and is not corrected for.

**Practical**

- `get_stream_id_from_label("emg")` is not a presence check on Gen 2. Check the record count too.
- `emg_calibration` and `imu_calibration` are both optional. Check before use.
- `adc_to_volts` raises on out-of-range input only when `is_truncated()` is `True`; otherwise it
  extrapolates silently.
- Convert a whole flat list in one `adc_to_volts` call and reshape; do not call it per sample.
- Plot with `rr.send_columns`, not a per-sample `rr.log`.

**Related tutorials**

- `Tutorial_1_vrs_data_provider_basics` — the query APIs used throughout
- `Tutorial_3_sequential_access_multi_sensor_data` — the queued delivery API
- `Tutorial_6_timestamp_alignment_in_aria_gen2` — the time domains the device clock sits among
- `Tutorial_8_eyetracking` — the other Gen 2 signal with a source-specific calibration story